# Regional expression of brain-disorder risk genes: full analysis

One notebook, run top to bottom. It rebuilds the analysis from the Allen Human Brain Atlas expression data and the GWAS Catalog, and writes everything to a separate `repro` folder on Google Drive, so nothing from your original analysis is overwritten.

**How to run**
1. Colab: *Runtime > Disconnect and delete runtime*, then *Runtime > Run all*. Allow the Google Drive permission.
2. First do a quick test: set `QUICK_TEST = True` in the first code cell (about 10 minutes, fewer permutations, three diseases). If it runs to the end, set it back to `False` and run all again for the real thing.
3. The full run takes roughly 1 to 3 hours. Every long step saves its result and is skipped if the file already exists, so if Colab disconnects, run all again and it continues.

**What it does:** (1) builds the 83-region expression table with abagen; (2) builds gene lists for 11 disorders and 2 controls from the GWAS Catalog; (3) scores each region against random gene sets, including gradient-matched nulls; (4) makes the figures and summary tables; (5) compares the results with the ones in the paper.

**Not included:** the first Alzheimer's variants (free-text trait names) and the Huntington's GWAS download. Their results are in `docs/notes.md`.

abagen 0.1.3 predates pandas 2, so section 2 applies two small compatibility fixes. They do not change the computation, but the output has not been compared with an unpatched environment.

## 0. Settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, re, glob, warnings
warnings.filterwarnings('ignore')

PROJECT = '/content/drive/MyDrive/hd_project'   # change if your Drive folder has a different name
QUICK_TEST = False                              # True = fast test run with fewer permutations
N_PERM = 200 if QUICK_TEST else 10000
OUT = f'{PROJECT}/repro_quick' if QUICK_TEST else f'{PROJECT}/repro'
DATA_DIR = f'{PROJECT}/abagen_data'
EXPR_FILE = f'{OUT}/expression_dk_named.csv'
for d in [OUT, f'{OUT}/figures', DATA_DIR]:
    os.makedirs(d, exist_ok=True)
print('output folder:', OUT, '| permutations:', N_PERM)

## 1. Expression table

If the table already exists in the output folder, this section only loads it. Otherwise it builds the table, which is the slow part of a first run (about 1.5 GB of donor data; resumes if interrupted).

abagen 0.1.3 only works with older library versions than a current Colab has, so the build runs in a **separate, isolated environment** with the versions used for the original analysis (numpy 2.1.3, pandas 2.2.3, scipy 1.16.3, nibabel 5.4.2). That keeps the rest of the notebook untouched and needs no restart. Compatibility fixes are applied to abagen inside that environment: `set_axis(..., inplace=False)`, `np.row_stack`, a one-element-array return in the tie-break step, and a stand-in for `DataFrame.append`. None of them changes what the code computes, but the output has not been compared with an unpatched run.

In [ ]:
NEED_BUILD = not os.path.exists(EXPR_FILE)
print('will build the expression table' if NEED_BUILD else 'expression table found: loading it')
ENV_DIR = '/content/abagen_env'

if NEED_BUILD:
    import subprocess
    if not os.path.isdir(f'{ENV_DIR}/abagen'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--target', ENV_DIR,
                        'numpy==2.1.3', 'pandas==2.2.3', 'scipy==1.16.3', 'nibabel==5.4.2',
                        'setuptools<81', 'abagen==0.1.3'], check=True)

    # compatibility patches, applied to the isolated copy of abagen only
    patched = []
    for f in glob.glob(ENV_DIR + '/abagen/**/*.py', recursive=True):
        t = open(f).read()
        new = re.sub(r'(set_axis\([^)]*?),\s*inplace=False\s*\)', r'\1)', t)
        new = re.sub(r'\bnp\.row_stack\b', 'np.vstack', new)
        new = new.replace('return labels[closest_centroid(coords, centroids)]',
                          'return labels[closest_centroid(coords, centroids)[0]]')
        if new != t:
            open(f, 'w').write(new)
            patched.append(os.path.basename(f))
    print('patched files:', sorted(set(patched)))
    probes_src = open(f'{ENV_DIR}/abagen/probes_.py').read()
    assert not re.search(r'set_axis\([^)]*inplace', probes_src), 'set_axis patch did not apply'
    assert 'np.row_stack' not in open(f'{ENV_DIR}/abagen/matching.py').read(), 'row_stack patch did not apply'

In [ ]:
if NEED_BUILD:
    BUILD_SCRIPT = """
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
print('building with numpy', np.__version__, '| pandas', pd.__version__, flush=True)
DATA_DIR, EXPR_FILE = sys.argv[1], sys.argv[2]

def _df_append(self, other, ignore_index=False, verify_integrity=False, sort=False):
    if isinstance(other, pd.Series):
        other = other.to_frame().T
    if isinstance(other, (list, tuple)):
        frames = [self] + [o.to_frame().T if isinstance(o, pd.Series) else o for o in other]
    else:
        frames = [self, other]
    return pd.concat(frames, ignore_index=ignore_index, verify_integrity=verify_integrity, sort=sort)
pd.DataFrame.append = _df_append

import abagen
print('downloading donor data (skipped if already on Drive)...', flush=True)
abagen.fetch_microarray(data_dir=DATA_DIR, donors='all')
atlas = abagen.fetch_desikan_killiany()
print('building the expression table (this takes a while)...', flush=True)
expression = abagen.get_expression_data(atlas['image'], atlas['info'],
                                        lr_mirror='bidirectional',
                                        data_dir=DATA_DIR)
info = pd.read_csv(atlas['info'])
names = dict(zip(info['id'], info['hemisphere'] + '_' + info['label']))
expression.index = expression.index.map(names)
expression.index.name = 'label'
expression.to_csv(EXPR_FILE)
print('saved', expression.shape, flush=True)
"""
    open('/content/build_expression.py', 'w').write(BUILD_SCRIPT)
    env = dict(os.environ, PYTHONPATH=ENV_DIR)
    proc = subprocess.Popen([sys.executable, '/content/build_expression.py', DATA_DIR, EXPR_FILE],
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if 'Downloaded' in line and '100.0%' not in line:
            continue
        print(line, end='')
    assert proc.wait() == 0, 'the build failed: send the output above'

In [ ]:
import numpy as np, pandas as pd
expression = pd.read_csv(EXPR_FILE, index_col=0)

print('shape:', expression.shape, '(expected 83 rows, about 15,633 genes)')
print('NaNs:', int(expression.isna().sum().sum()))
corr = expression.T.corr()
left = [r for r in expression.index if r.startswith('L_')]
pairs = [(r, 'R_' + r[2:]) for r in left if 'R_' + r[2:] in expression.index]
print('mean L-vs-R correlation:', round(float(np.mean([corr.loc[a, b] for a, b in pairs])), 2), '(expected about 0.98)')
for g in ['APOE', 'APP', 'BIN1', 'MAPT', 'TREM2', 'HTT']:
    assert g in expression.columns, g + ' missing'

# all statistics use the left hemisphere plus brainstem (42 regions); the right side is a mirrored copy
expr_L = expression.loc[[r for r in expression.index if r.startswith(('L_', 'B_'))]]
X = expr_L.values
cols = np.array(expr_L.columns)
print('analysis table:', expr_L.shape)

orig = f'{PROJECT}/expression_dk_named.csv'
if os.path.exists(orig) and orig != EXPR_FILE:
    old = pd.read_csv(orig, index_col=0)
    if old.shape == expression.shape:
        diff = np.abs(old.values - expression.loc[old.index, old.columns].values).max()
        print('max absolute difference from the table used for the paper:', float(diff))
    else:
        print('shape differs from the table used for the paper:', old.shape)

## 2. Statistical functions

- `score`: mean expression of a gene list per region, compared with random gene sets of the same size; z-score, one-sided permutation p = (count + 1) / (n + 1), Benjamini-Hochberg FDR across the 42 regions. Real list genes are excluded from the random draws; seed 0.
- `score_binned`: the same, but random genes are drawn from bins of equal loading on the first expression principal component (20 bins) or on the first two (6 x 6 bins), so the null follows the dominant expression gradients.
- `score_roi`: one test on the mean across several pre-named regions.

In [ ]:
from statsmodels.stats.multitest import multipletests

alias = {'CIAO2A': 'FAM96A'}      # old name present in the expression table
def present(genes):
    out = []
    for g in genes:
        if g in expr_L.columns:
            out.append(g)
        elif g in alias and alias[g] in expr_L.columns:
            out.append(alias[g])
    return sorted(set(out))

def _finish(obs, null):
    z = (obs - null.mean(0)) / null.std(0)
    p = ((null >= obs).sum(0) + 1) / (null.shape[0] + 1)
    q = multipletests(p, method='fdr_bh')[1]
    return pd.DataFrame({'z': z, 'p': p, 'q_fdr': q}, index=expr_L.index)

def score(genes, n_perm=None, seed=0):
    n_perm = n_perm or N_PERM
    sel = present(genes)
    sel_idx = np.where(np.isin(cols, sel))[0]
    pool = np.setdiff1d(np.arange(len(cols)), sel_idx)
    rng = np.random.default_rng(seed)
    obs = X[:, sel_idx].mean(axis=1)
    null = np.empty((n_perm, X.shape[0]))
    for i in range(n_perm):
        null[i] = X[:, rng.choice(pool, size=len(sel), replace=False)].mean(axis=1)
    return _finish(obs, null), len(sel)

# principal axes of regional expression
Ec = ((expr_L - expr_L.mean()) / expr_L.std()).fillna(0)
_, sv, vt = np.linalg.svd(Ec.values, full_matrices=False)
print('variance explained by PC1-3:', (sv[:3] ** 2 / (sv ** 2).sum()).round(3))

def make_bins(k1, k2=None):
    b1 = pd.qcut(pd.Series(vt[0]).rank(method='first'), k1, labels=False).values
    if k2 is None:
        return b1, k1
    b2 = pd.qcut(pd.Series(vt[1]).rank(method='first'), k2, labels=False).values
    return b1 * k2 + b2, k1 * k2

bins_1d, nb_1d = make_bins(20)
bins_2d, nb_2d = make_bins(6, 6)

def score_binned(genes, bins, n_bins, n_perm=None, seed=0):
    n_perm = n_perm or N_PERM
    sel = present(genes)
    sel_idx = np.where(np.isin(cols, sel))[0]
    in_sel = np.zeros(X.shape[1], bool)
    in_sel[sel_idx] = True
    need = np.bincount(bins[sel_idx], minlength=n_bins)
    pools = [np.where((bins == b) & ~in_sel)[0] for b in range(n_bins)]
    assert all(len(pools[b]) >= need[b] for b in range(n_bins)), 'a bin has too few genes'
    active = [b for b in range(n_bins) if need[b] > 0]
    rng = np.random.default_rng(seed)
    obs = X[:, sel_idx].mean(axis=1)
    null = np.empty((n_perm, X.shape[0]))
    for i in range(n_perm):
        idx = np.concatenate([rng.choice(pools[b], size=need[b], replace=False) for b in active])
        null[i] = X[:, idx].mean(axis=1)
    return _finish(obs, null)

def score_roi(genes, rois, n_perm=None, seed=0):
    n_perm = n_perm or N_PERM
    sel = present(genes)
    sel_idx = np.where(np.isin(cols, sel))[0]
    pool = np.setdiff1d(np.arange(len(cols)), sel_idx)
    ridx = [list(expr_L.index).index(r) for r in rois]
    Xr = X[ridx]
    rng = np.random.default_rng(seed)
    obs = Xr[:, sel_idx].mean()
    null = np.array([Xr[:, rng.choice(pool, size=len(sel), replace=False)].mean() for _ in range(n_perm)])
    return round(float((obs - null.mean()) / null.std()), 3), round(float(((null >= obs).sum() + 1) / (n_perm + 1)), 4)

## 3. Positive control: striatal marker genes

Ten markers chosen before the analysis (nine are present in the expression table). If the method works, caudate and putamen should score highest.

In [ ]:
striatal = ['PPP1R1B', 'DRD1', 'DRD2', 'ADORA2A', 'GPR88', 'PDE10A', 'RGS9', 'PENK', 'TAC1', 'RASD2']
f = f'{OUT}/control_striatal.csv'
if os.path.exists(f):
    res_str = pd.read_csv(f, index_col=0)
else:
    res_str, k = score(striatal)
    res_str.to_csv(f)
print('usable markers:', present(striatal))
print(res_str.sort_values('z', ascending=False).head(5).round(4))

## 4. Gene lists from the GWAS Catalog

Uses `gwas_slim.csv.gz` from your Drive folder if it is there (the exact file used for the paper, downloaded 2 November 2026). Otherwise it downloads the current full catalog, **which changes with every release, so counts may differ slightly from the paper**.

Rules, fixed before looking at results: p < 5e-8; `MAPPED_TRAIT` must equal the disease label exactly (no multi-trait or polygenic-score rows); 20 to 500 usable genes (more than 500: keep the 500 strongest; fewer than 20: excluded); a second "clean" list drops the MHC (chr6 25-34 Mb) and APOE (chr19 44.4-45.4 Mb) regions. Coordinates assume genome build GRCh38.

In [ ]:
import subprocess, zipfile
SLIM_ORIG = f'{PROJECT}/gwas_slim.csv.gz'
SLIM_NEW = f'{OUT}/gwas_slim.csv.gz'

if os.path.exists(SLIM_ORIG):
    slim_path = SLIM_ORIG
    print('using the GWAS file from the paper:', SLIM_ORIG)
elif os.path.exists(SLIM_NEW):
    slim_path = SLIM_NEW
else:
    print('downloading the current GWAS Catalog (counts may differ from the paper)')
    base = 'https://ftp.ebi.ac.uk/pub/databases/gwas/releases/latest/'
    fname = 'gwas-catalog-associations_ontology-annotated-full.zip'
    subprocess.run(['wget', '-q', '-O', f'/content/{fname}', base + fname], check=True)
    with zipfile.ZipFile(f'/content/{fname}') as z:
        z.extractall('/content/gwas_full')
    tsv = [x for x in os.listdir('/content/gwas_full') if x.endswith('.tsv')][0]
    want = ['DISEASE/TRAIT', 'MAPPED_TRAIT', 'MAPPED_TRAIT_URI', 'MAPPED_GENE', 'REPORTED GENE(S)',
            'P-VALUE', 'CHR_ID', 'CHR_POS', 'STUDY ACCESSION', 'PUBMEDID', 'INTERGENIC', 'SNPS']
    header = pd.read_csv(f'/content/gwas_full/{tsv}', sep='\t', nrows=0).columns.tolist()
    big_ = pd.read_csv(f'/content/gwas_full/{tsv}', sep='\t', usecols=[c for c in want if c in header],
                       dtype=str, low_memory=False)
    big_.to_csv(SLIM_NEW, index=False, compression='gzip')
    slim_path = SLIM_NEW

big = pd.read_csv(slim_path, dtype=str)
big['P-VALUE'] = pd.to_numeric(big['P-VALUE'], errors='coerce')
big['pos'] = pd.to_numeric(big['CHR_POS'], errors='coerce')
big['mt'] = big['MAPPED_TRAIT'].str.lower().str.strip()
sig = big[big['P-VALUE'] < 5e-8].copy()
print(big.shape[0], 'associations;', len(sig), 'with p < 5e-8')

In [ ]:
diseases = {
    'alzheimer': ['alzheimer disease'],
    'parkinson': ['parkinson disease'],
    'als': ['amyotrophic lateral sclerosis', 'sporadic amyotrophic lateral sclerosis'],
    'ms': ['multiple sclerosis'],
    'schizophrenia': ['schizophrenia'],
    'bipolar': ['bipolar disorder'],
    'mdd': ['major depressive disorder'],
    'adhd': ['attention deficit-hyperactivity disorder'],
    'autism': ['autism spectrum disorder'],
    'epilepsy': ['epilepsy', 'idiopathic generalized epilepsy', 'focal epilepsy'],
    'migraine': ['migraine disorder'],
    'crohn': ['crohn disease'],
    'cad': ['coronary artery disorder'],
}
if QUICK_TEST:
    diseases = {k: v for k, v in diseases.items() if k in ('alzheimer', 'schizophrenia', 'crohn')}

def excluded_region(df):
    mhc = (df['CHR_ID'] == '6') & df['pos'].between(25e6, 34e6)
    apoe = (df['CHR_ID'] == '19') & df['pos'].between(44.4e6, 45.4e6)
    return mhc | apoe

def genes_ranked(df):
    rows = []
    for g, p in zip(df['MAPPED_GENE'], df['P-VALUE']):
        if isinstance(g, str):
            for x in re.split(r'\s+-\s+|\s*,\s*|\s*;\s*|\s+x\s+', g):
                x = x.strip()
                if x:
                    rows.append((x, p))
    if not rows:
        return []
    s = pd.DataFrame(rows, columns=['gene', 'p']).groupby('gene')['p'].min().sort_values(kind='stable')
    return [g for g in s.index if present([g])]

def decide(n):
    return 'EXCLUDE (<20)' if n < 20 else ('cap to 500' if n > 500 else 'ok')

plan, usable = [], {}
for name, labels in diseases.items():
    d = sig[sig['mt'].isin(labels)]
    for version, dd in (('all', d), ('clean', d[~excluded_region(d)])):
        genes = genes_ranked(dd)
        plan.append({'disease': name, 'version': version, 'rows': len(dd), 'usable_genes': len(genes),
                     'decision': decide(len(genes))})
        if len(genes) >= 20:
            usable[(name, version)] = genes[:500]
            pd.Series(genes[:500]).to_csv(f'{OUT}/grid_{name}_{version}_genes.csv', index=False, header=['gene'])
plan = pd.DataFrame(plan)
plan.to_csv(f'{OUT}/grid_plan.csv', index=False)
print(plan.to_string(index=False))

## 5. The grid: every disease, every null model

Four versions per disease (counts of significant regions are compared in the summary below):
1. standard random-gene null, full list
2. standard null, list without the MHC and APOE regions
3. PC1-matched null, full list (**the primary analysis**)
4. PC1-matched null, clean list
5. PC1 x PC2-matched null, full list

This is the long step. Results are saved one file at a time and skipped if they already exist.

In [ ]:
def run_one(fn, genes, kind):
    path = f'{OUT}/{fn}'
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0)
    if kind == 'standard':
        res, _ = score(genes)
    elif kind == 'pc1':
        res = score_binned(genes, bins_1d, nb_1d)
    else:
        res = score_binned(genes, bins_2d, nb_2d)
    res.to_csv(path)
    print('done:', fn)
    return res

R = {}
for name in diseases:
    for version in ('all', 'clean'):
        if (name, version) not in usable:
            print(f'{name}/{version}: excluded ({int(plan[(plan.disease == name) & (plan.version == version)].usable_genes.iloc[0])} usable genes)')
            continue
        g = usable[(name, version)]
        R[('standard', name, version)] = run_one(f'grid_{name}_{version}.csv', g, 'standard')
        if version == 'all':
            R[('pc1', name, 'all')] = run_one(f'grid_matched_{name}_all.csv', g, 'pc1')
            R[('pc2d', name, 'all')] = run_one(f'grid_matched2d_{name}_all.csv', g, 'pc2d')
        else:
            R[('pc1', name, 'clean')] = run_one(f'grid_matchedclean_{name}_clean.csv', g, 'pc1')
print('grid finished')

## 6. Summary tables

In [ ]:
analysed = [n for n in diseases if ('standard', n, 'all') in R]

rows = []
for n in analysed:
    row = {'disease': n, 'usable_genes': len(usable[(n, 'all')]),
           'standard': int((R[('standard', n, 'all')]['q_fdr'] < 0.05).sum()),
           'PC1_matched_primary': int((R[('pc1', n, 'all')]['q_fdr'] < 0.05).sum()),
           'PC1_matched_clean': int((R[('pc1', n, 'clean')]['q_fdr'] < 0.05).sum()) if ('pc1', n, 'clean') in R else None,
           'PC1xPC2_matched': int((R[('pc2d', n, 'all')]['q_fdr'] < 0.05).sum()),
           'best_z_PC1_matched': round(float(R[('pc1', n, 'all')]['z'].max()), 2),
           'best_q_PC1_matched': round(float(R[('pc1', n, 'all')]['q_fdr'].min()), 3)}
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(f'{OUT}/grid_summary.csv', index=False)
print('regions with q < 0.05 (of 42) under each null:')
print(summary.to_string(index=False))

# pre-declared region tests (secondary, exploratory; standard null)
predicted = {
    'alzheimer': ['L_entorhinal', 'L_hippocampus'],
    'als': ['L_precentral'],
    'epilepsy': ['L_hippocampus', 'L_entorhinal', 'L_parahippocampal', 'L_temporalpole'],
    'schizophrenia': ['L_hippocampus', 'L_superiorfrontal', 'L_rostralmiddlefrontal', 'L_caudalmiddlefrontal'],
}
print('\npre-declared region tests (z, p):')
for n, regs in predicted.items():
    if n in analysed:
        print(n, score_roi(usable[(n, 'all')], [r for r in regs if r in expr_L.index]))

In [ ]:
# shared expression gradient and overlap between gene lists
Z = pd.DataFrame({n: R[('standard', n, 'all')]['z'] for n in analysed})
pc1 = pd.Series(np.linalg.svd(Ec.values, full_matrices=False)[0][:, 0], index=expr_L.index)
print('correlation of each disease z-profile (standard null) with PC1:')
print(Z.apply(lambda c: c.corr(pc1)).round(2).to_string())

lists = {n: set(usable[(n, 'all')]) for n in analysed}
ov = pd.DataFrame({a: {b: len(lists[a] & lists[b]) / len(lists[a] | lists[b]) for b in analysed} for a in analysed})
print('\ngene-list overlap (Jaccard):')
print(ov.round(2).to_string())

print('\nHuntington (descriptive, no test): rank of caudate / putamen among 42 regions')
for g in ['HTT', 'FAN1']:
    if g in expr_L.columns:
        rk = expr_L[g].rank(ascending=False)
        print(g, int(rk['L_caudate']), '/', int(rk['L_putamen']))

## 7. Figures

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

Zn = pd.DataFrame({n: R[('standard', n, 'all')]['z'] for n in analysed})
Zm = pd.DataFrame({n: R[('pc1', n, 'all')]['z'] for n in analysed})

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
for ax, Zm_, title in zip(axes, [Zn, Zm], ['Standard random-gene null', 'PC1-matched null']):
    im = ax.imshow(Zm_.T.values, aspect='auto', cmap='RdBu_r', vmin=-6, vmax=6)
    ax.set_yticks(range(len(analysed))); ax.set_yticklabels(analysed); ax.set_title(title)
axes[1].set_xticks(range(len(Zm.index))); axes[1].set_xticklabels(Zm.index, rotation=90, fontsize=7)
fig.colorbar(im, ax=axes, label='z-score')
plt.savefig(f'{OUT}/figures/fig1_heatmaps.png', dpi=200, bbox_inches='tight'); plt.show()

counts = summary.set_index('disease')[['standard', 'PC1_matched_primary', 'PC1_matched_clean', 'PC1xPC2_matched']]
counts.plot.bar(figsize=(13, 5))
plt.ylabel('regions with q < 0.05 (of 42)'); plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig2_robustness.png', dpi=200); plt.show()

sc = res_str.sort_values('z', ascending=False)
colors = ['#c0392b' if r in ('L_caudate', 'L_putamen', 'L_accumbensarea') else '#95a5a6' for r in sc.index]
plt.figure(figsize=(14, 4)); plt.bar(range(len(sc)), sc['z'], color=colors)
plt.xticks(range(len(sc)), sc.index, rotation=90, fontsize=7)
plt.ylabel('z-score'); plt.title('Positive control: striatal marker genes'); plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig3_striatal_control.png', dpi=200); plt.show()

if len(analysed) >= 3:
    plt.figure(figsize=(8, 4))
    dendrogram(linkage(Zm.T.values, method='average', metric='correlation'), labels=analysed, leaf_rotation=60)
    plt.ylabel('correlation distance'); plt.tight_layout()
    plt.savefig(f'{OUT}/figures/fig4_clustering.png', dpi=200); plt.show()

## 8. Compare with the results in the paper

For every result file, the largest absolute difference in z-score between this run and the original analysis. **Meaningful only when `QUICK_TEST = False`.** Differences near 0 mean the analysis reproduces from scratch. If the GWAS file was downloaded fresh (not the paper's copy), small differences are expected.

In [ ]:
patterns = ['grid_{n}_all.csv', 'grid_{n}_clean.csv', 'grid_matched_{n}_all.csv',
            'grid_matchedclean_{n}_clean.csv', 'grid_matched2d_{n}_all.csv']
rows, missing = [], 0
for n in analysed:
    for p in patterns:
        a, b = f'{PROJECT}/' + p.format(n=n), f'{OUT}/' + p.format(n=n)
        if os.path.exists(a) and os.path.exists(b):
            da, db = pd.read_csv(a, index_col=0), pd.read_csv(b, index_col=0)
            rows.append({'file': p.format(n=n),
                         'max_abs_z_diff': float(np.abs(da['z'].values - db['z'].values).max()),
                         'sig_regions_original': int((da['q_fdr'] < 0.05).sum()),
                         'sig_regions_this_run': int((db['q_fdr'] < 0.05).sum())})
        else:
            missing += 1
cmp_ = pd.DataFrame(rows)
if QUICK_TEST:
    print('QUICK_TEST is on: this run used', N_PERM, 'permutations instead of 10,000, so differences from the paper are expected.')
    print('Set QUICK_TEST = False and run all again for a real comparison.')
elif len(cmp_) == 0:
    print('no original files to compare with')
else:
    worst = cmp_.loc[cmp_['max_abs_z_diff'].idxmax()]
    changed = cmp_[cmp_['sig_regions_original'] != cmp_['sig_regions_this_run']]
    print('files compared:', len(cmp_), '| largest difference in z:', round(worst['max_abs_z_diff'], 6), 'in', worst['file'])
    print('files where the number of significant regions changed:', len(changed))
    if len(changed):
        print(changed.to_string(index=False))
    if worst['max_abs_z_diff'] < 1e-6:
        print('REPRODUCED EXACTLY')
    elif worst['max_abs_z_diff'] < 0.1 and len(changed) == 0:
        print('REPRODUCED WITHIN SIMULATION NOISE (random draws can differ slightly if the order of expression bins differs)')
    else:
        print('DIFFERENCES FOUND: check the GWAS file version and the expression table comparison in section 1')
print('files with no counterpart:', missing)
print('\nresults are in:', OUT)